# Stage 3–4: POIs → buildings → floors
**Inputs:** your OSM POI layer (shapefile or GeoPackage), the study-area shapefile, and the buildings file from the heights notebook (`buildings_floors_<AREA>.gpkg`).
**Outputs:** every POI with a land-use class, its building ID, its floor (where known) and the source of that floor; summary figures.

Change only **Cell 1 (Settings)**.

## 1. Settings

In [ ]:
import os, re, json
import numpy as np, pandas as pd, geopandas as gpd
from IPython.display import display

POI_FILE  = r'path/to/osm_points.shp'                         # <-- your OSM points layer (.shp or .gpkg)
SHP       = r'path/to/study_area.shp'    # study area (here: 225 BBMP wards)
WARD_COL  = 'name_en'      # ward-name column in SHP; None if the boundary has no wards
AREA_NAME = 'bengaluru'
OUT_DIR   = r'outputs'
BLDG_FILE = os.path.join(OUT_DIR, f'buildings_floors_{AREA_NAME}.gpkg')

MATCH_DIST_M = 20          # POIs outside every footprint snap to the nearest building within this distance

FIG_DIR = os.path.join(OUT_DIR, f'figures_{AREA_NAME}')
os.makedirs(FIG_DIR, exist_ok=True)

### Chart style

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Patch

INK, INK2, MUTED, GRID, AXIS, SURFACE = '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7', '#fcfcfb'
BLUE, LIGHT_BLUE, NODATA = '#2a78d6', '#86b6ef', '#c3c2b7'
SEQ = ['#cde2fb', '#9ec5f4', '#6da7ec', '#3987e5', '#256abf', '#184f95', '#0d366b']

mpl.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Segoe UI', 'Arial', 'DejaVu Sans'],
    'font.size': 10, 'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'text.color': INK, 'axes.labelcolor': INK2, 'xtick.color': MUTED, 'ytick.color': MUTED,
    'axes.edgecolor': AXIS, 'axes.linewidth': 0.8, 'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.6,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
    'savefig.dpi': 200, 'savefig.bbox': 'tight'})

def save(fig, name):
    base = os.path.join(FIG_DIR, os.path.splitext(name)[0])
    fig.savefig(base + '.png')                 # raster, 200 dpi
    fig.savefig(base + '.svg')                 # vector: text, axes and outlines stay editable
    print('saved', base + '.png / .svg')

## 2. Load POIs and read OSM tags
OSM exports from QGIS keep most tags in one text column, `other_tags`. This cell splits it into proper columns.
Shapefiles cut that column at 254 characters, so a few tags can be lost; exporting the layer as **GeoPackage** avoids that.

In [ ]:
if POI_FILE.lower().endswith('.shp'):
    poi = gpd.read_file(POI_FILE, encoding='latin1')           # read raw bytes, then repair UTF-8 text
    fix = lambda s: s.encode('latin1').decode('utf-8', 'ignore') if isinstance(s, str) else s
    for c in poi.columns:
        if c != 'geometry' and pd.api.types.is_string_dtype(poi[c]):
            poi[c] = poi[c].map(fix)
else:
    poi = gpd.read_file(POI_FILE)
if poi.crs is None:
    poi = poi.set_crs(4326)
poi = poi.to_crs(4326)
print('Points loaded:', len(poi))

KV = re.compile(r'"([^"]+)"=>"((?:[^"\\]|\\.)*)"')
tags = poi.get('other_tags', pd.Series('', index=poi.index)).fillna('').map(lambda s: dict(KV.findall(s)))
print('other_tags cut off at 254 characters:', (poi.get('other_tags', pd.Series('')).fillna('').str.len() >= 250).sum())

TAG_KEYS = ['amenity', 'shop', 'office', 'tourism', 'leisure', 'healthcare', 'craft', 'sport',
            'level', 'addr:floor', 'addr:unit', 'addr:full', 'addr:housenumber', 'addr:street', 'brand', 'cuisine']
for k in TAG_KEYS:
    col = k.replace(':', '_')
    if col not in poi.columns:
        poi[col] = tags.map(lambda d: d.get(k))

## 3. Clip to the study area

In [ ]:
aoi = gpd.read_file(SHP).to_crs(4326)
aoi_geom = (aoi.geometry.union_all() if hasattr(aoi.geometry, 'union_all') else aoi.geometry.unary_union).buffer(0)
UTM = aoi.estimate_utm_crs()
poi = poi[poi.within(aoi_geom)].copy()
print('Points inside study area:', len(poi))
if WARD_COL and WARD_COL in aoi.columns:
    w = gpd.sjoin(poi[['geometry']], aoi[[WARD_COL, 'geometry']], how='left', predicate='within')
    poi['ward'] = w[~w.index.duplicated()][WARD_COL]
    print('Wards:', aoi[WARD_COL].nunique(), '| POIs without a ward:', poi['ward'].isna().sum())

## 4. Classify land use
OSM tags are grouped into ten land-use classes. Street furniture (benches, bins, ATMs, parking, toilets...) is removed because it does not occupy a floor of a building.

In [ ]:
DROP_AMENITY = {'bench', 'waste_basket', 'toilets', 'parking', 'parking_entrance', 'parking_space', 'bicycle_parking',
                'motorcycle_parking', 'drinking_water', 'vending_machine', 'atm', 'telephone', 'post_box', 'clock',
                'fountain', 'shelter', 'recycling', 'waste_disposal', 'charging_station', 'taxi', 'bus_station',
                'water_point', 'watering_place', 'grave_yard', 'hunting_stand', 'loading_dock', 'compressed_air',
                'letter_box', 'bbq', 'feeding_place', 'smoking_area', 'public_bookcase', 'give_box', 'device_charging_station'}
FOOD     = {'restaurant', 'fast_food', 'cafe', 'bar', 'pub', 'food_court', 'ice_cream', 'biergarten', 'juice_bar'}
HEALTH   = {'hospital', 'clinic', 'doctors', 'dentist', 'pharmacy', 'veterinary', 'nursing_home'}
EDU      = {'school', 'college', 'university', 'kindergarten', 'library', 'training', 'language_school',
            'driving_school', 'music_school', 'research_institute', 'childcare', 'prep_school', 'tuition'}
FINANCE  = {'bank', 'bureau_de_change', 'money_transfer', 'payment_centre'}
CIVIC    = {'police', 'townhall', 'courthouse', 'fire_station', 'post_office', 'community_centre', 'social_facility',
            'public_building', 'embassy', 'prison'}
LEISURE_A = {'cinema', 'theatre', 'nightclub', 'arts_centre', 'events_venue', 'casino', 'studio', 'conference_centre'}
WORSHIP  = {'place_of_worship', 'monastery'}
PERSONAL_SHOP = {'hairdresser', 'beauty', 'laundry', 'dry_cleaning', 'tailor', 'massage', 'tattoo', 'optician',
                 'travel_agency', 'copyshop', 'mobile_phone_repair', 'funeral_directors'}
FOOD_SHOP = {'bakery', 'pastry', 'confectionery', 'coffee', 'tea'}

def classify(r):
    a, s, o, t, l, h = r.amenity, r.shop, r.office, r.tourism, r.leisure, r.healthcare
    if isinstance(a, str) and a in DROP_AMENITY: return None
    if isinstance(o, str):
        return 'Civic' if o in {'government', 'diplomatic'} else 'Office'
    if isinstance(a, str):
        if a in FOOD: return 'Food & drink'
        if a in HEALTH: return 'Health'
        if a in EDU: return 'Education'
        if a in FINANCE: return 'Finance'
        if a in CIVIC: return 'Civic'
        if a in LEISURE_A: return 'Leisure'
        if a in WORSHIP: return 'Religious'
        if a == 'coworking_space': return 'Office'
    if isinstance(h, str): return 'Health'
    if isinstance(t, str) and t in {'hotel', 'guest_house', 'hostel', 'motel', 'apartment'}: return 'Hospitality'
    if isinstance(l, str) and l in {'fitness_centre', 'sports_centre', 'dance', 'bowling_alley', 'amusement_arcade',
                                     'escape_game', 'sports_hall', 'trampoline_park'}: return 'Leisure'
    if isinstance(s, str):
        if s in PERSONAL_SHOP: return 'Personal services'
        if s in FOOD_SHOP: return 'Food & drink'
        return 'Retail'
    if isinstance(r.craft, str): return 'Personal services'
    return None

poi['landuse'] = poi.apply(classify, axis=1)
LANDUSE_ORDER = ['Retail', 'Food & drink', 'Office', 'Personal services', 'Health', 'Finance',
                 'Education', 'Hospitality', 'Leisure', 'Civic', 'Religious']
poi = poi[poi['landuse'].notna()].copy()
poi['category'] = (poi['amenity'].fillna(poi['shop']).fillna(poi['office']).fillna(poi['tourism'])
                   .fillna(poi['leisure']).fillna(poi['healthcare']).fillna(poi['craft']))
print('POIs kept:', len(poi))
display(poi['landuse'].value_counts().reindex(LANDUSE_ORDER).dropna().astype(int).to_frame('POIs'))

## 5. Read the floor of each POI
Two sources, in this order:
1. OSM tags `level` / `addr:floor` (in OSM, `level=0` is the ground floor, `-1` the basement).
2. Floors written in the name or address: *"3rd Floor"*, *"First Flr"*, *"GF"*, *"UGF"*, *"Basement"*...

Floors are stored as numbers: ground = 0, first = 1, basement = −1.

In [ ]:
WORD_NUM = {'ground': 0, 'first': 1, 'second': 2, 'third': 3, 'fourth': 4, 'fifth': 5, 'sixth': 6,
            'seventh': 7, 'eighth': 8, 'ninth': 9, 'tenth': 10, 'eleventh': 11, 'twelfth': 12}
RX_NUM   = re.compile(r'\b(\d{1,2})\s*(?:st|nd|rd|th)?\s*(?:floor|flr|fl)\b', re.I)
RX_WORD  = re.compile(r'\b(' + '|'.join(WORD_NUM) + r')\s*(?:floor|flr|fl)\b', re.I)
RX_FLOOR_N = re.compile(r'\bfloor\s*(?:no\.?\s*)?(\d{1,2})\b', re.I)
RX_CODES = {'ugf': 0, 'upper ground': 0, 'gf': 0, 'grd': 0, 'lgf': -1, 'lower ground': -1,
            'basement': -1, 'cellar': -1, 'mezzanine': 0, 'terrace': None}
RX_CODE  = re.compile(r'\b(' + '|'.join(sorted(RX_CODES, key=len, reverse=True)) + r')\b', re.I)

def parse_level_tag(v):
    if not isinstance(v, str) or not v.strip():
        return np.nan
    nums = re.findall(r'-?\d+(?:\.\d+)?', v)              # '0;1' or '0-2' -> lowest floor
    return float(min(float(n) for n in nums)) if nums else np.nan

def parse_text(s):
    if not isinstance(s, str) or not s.strip():
        return np.nan
    for rx in (RX_NUM, RX_FLOOR_N):
        m = rx.search(s)
        if m: return float(m.group(1))
    m = RX_WORD.search(s)
    if m: return float(WORD_NUM[m.group(1).lower()])
    m = RX_CODE.search(s)
    if m:
        v = RX_CODES[m.group(1).lower()]
        return np.nan if v is None else float(v)
    return np.nan

text_cols = [c for c in ['name', 'address', 'addr_full', 'addr_unit', 'addr_housenumber', 'addr_street'] if c in poi.columns]
poi['addr_text'] = poi[text_cols].fillna('').astype(str).agg(' | '.join, axis=1)

poi['floor_tag']  = poi['level'].map(parse_level_tag).fillna(poi['addr_floor'].map(parse_level_tag))
poi['floor_text'] = poi['addr_text'].map(parse_text)
poi['floor']        = poi['floor_tag'].fillna(poi['floor_text'])
poi['floor_source'] = np.select([poi['floor_tag'].notna(), poi['floor_text'].notna()], ['osm_tag', 'address_text'], 'unknown')

print(poi['floor_source'].value_counts())
print(f"Share of POIs with a floor: {poi['floor'].notna().mean():.1%}")
display(poi.loc[poi['floor_source'] == 'address_text', ['name', 'addr_text', 'floor']].head(10))

## 6. Link each POI to a building
First: the building whose footprint contains the POI. Otherwise: the nearest building within `MATCH_DIST_M` metres.

In [ ]:
bld = gpd.read_file(BLDG_FILE).to_crs(UTM)
pts = poi.to_crs(UTM)
print('Buildings:', len(bld))

inside = gpd.sjoin(pts, bld[['bid', 'geometry']], how='left', predicate='within')
inside = inside[~inside.index.duplicated()]                       # overlapping footprints: keep first
poi['bid'] = inside['bid']
poi['match'] = np.where(poi['bid'].notna(), 'inside', None)
poi['match_dist_m'] = np.where(poi['bid'].notna(), 0.0, np.nan)

miss = pts[poi['bid'].isna()]
if len(miss):
    near = gpd.sjoin_nearest(miss, bld[['bid', 'geometry']], how='left', max_distance=MATCH_DIST_M, distance_col='d')
    near = near[~near.index.duplicated()]
    ok = near['bid'].notna()
    poi.loc[near.index[ok], 'bid'] = near.loc[ok, 'bid']
    poi.loc[near.index[ok], 'match'] = 'nearest'
    poi.loc[near.index[ok], 'match_dist_m'] = near.loc[ok, 'd'].round(1)
poi['match'] = poi['match'].fillna('no_building')
print(poi['match'].value_counts())

# Bring in building floors and check parsed floors against them
bcols = ['bid', 'floors_final', 'floor_source', 'h_median', 'qc_flags']
poi = poi.merge(bld[bcols].rename(columns={'floor_source': 'bld_floor_source', 'qc_flags': 'bld_qc'}), on='bid', how='left')
poi['floor_above_building'] = poi['floor'].notna() & poi['floors_final'].notna() & (poi['floor'] >= poi['floors_final'] + 1)
print('POI floor higher than the building allows:', poi['floor_above_building'].sum())

## 7. Save

In [ ]:
keep = ['osm_id', 'name', 'ward', 'landuse', 'category', 'brand', 'addr_text', 'level', 'addr_floor',
        'floor', 'floor_source', 'floor_above_building', 'bid', 'match', 'match_dist_m',
        'floors_final', 'bld_floor_source', 'h_median', 'geometry']
keep = [c for c in keep if c in poi.columns]
out = gpd.GeoDataFrame(poi[keep], geometry='geometry', crs=4326)
out['lon'], out['lat'] = out.geometry.x.round(6), out.geometry.y.round(6)
out.to_file(os.path.join(OUT_DIR, f'poi_buildings_{AREA_NAME}.gpkg'), driver='GPKG')
out.drop(columns='geometry').to_csv(os.path.join(OUT_DIR, f'poi_buildings_{AREA_NAME}.csv'), index=False)
print('Saved', len(out), 'POIs')

---
# Visualisations

### P1. POIs by land use, and how many have a floor

In [ ]:
t = (poi.assign(has_floor=poi['floor'].notna())
        .groupby('landuse')['has_floor'].agg(total='size', with_floor='sum')
        .reindex(LANDUSE_ORDER).dropna())
t['without'] = t['total'] - t['with_floor']
t = t.sort_values('total')

fig, ax = plt.subplots(figsize=(9, 5.2))
ax.barh(t.index, t['with_floor'], color=BLUE, height=0.6, label='Floor known')
ax.barh(t.index, t['without'], left=t['with_floor'], color=LIGHT_BLUE, height=0.6, edgecolor=SURFACE, linewidth=2,
        label='Floor unknown')
for y, (tot, wf) in enumerate(zip(t['total'], t['with_floor'])):
    ax.text(tot, y, f'  {tot:,.0f}  ({wf / tot:.0%} known)', va='center', fontsize=8.5, color=INK2)
ax.set(title='POIs by land use', xlabel='POIs'); ax.grid(axis='y', visible=False); ax.margins(x=0.22)
ax.legend(frameon=False, loc='lower right')
ax.xaxis.set_major_formatter(mpl.ticker.StrMethodFormatter('{x:,.0f}'))
fig.tight_layout(); save(fig, 'p1_poi_landuse.png'); plt.show()

### P2. Floor information and building matches

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.2))
def stacked(ax, counts, order, labels, colors, title):
    left, tot = 0, counts.sum()
    for k, lab, c in zip(order, labels, colors):
        v = counts.get(k, 0)
        ax.barh(0, v, left=left, color=c, height=0.5, edgecolor=SURFACE, linewidth=2)
        if v / tot > 0.07:
            ax.text(left + v / 2, 0, f'{v / tot:.0%}', ha='center', va='center',
                    color='white' if c == BLUE else INK, fontsize=9)
        left += v
    ax.set(title=title, yticks=[], xlabel='POIs'); ax.grid(axis='y', visible=False)
    ax.legend(handles=[Patch(color=c, label=f'{lab} ({counts.get(k, 0):,})') for k, lab, c in zip(order, labels, colors)],
              loc='upper center', bbox_to_anchor=(0.5, -0.45), ncol=3, frameon=False, fontsize=8.5)

stacked(a1, poi['floor_source'].value_counts(), ['osm_tag', 'address_text', 'unknown'],
        ['OSM level tag', 'From address text', 'Unknown'], [BLUE, LIGHT_BLUE, NODATA], 'Where the POI floor comes from')
stacked(a2, poi['match'].value_counts(), ['inside', 'nearest', 'no_building'],
        ['Inside footprint', f'Nearest ≤ {MATCH_DIST_M} m', 'No building'], [BLUE, LIGHT_BLUE, NODATA], 'POI linked to a building')
fig.tight_layout(); save(fig, 'p2_floor_info_and_matching.png'); plt.show()

### P3. Which uses sit on which floors
Share of each land use's POIs on each floor (only POIs with a known floor).
This is the first look at the vertical pattern the entropy index will measure.

In [ ]:
fb = poi[poi['floor'].notna()].copy()
fb['floor_band'] = pd.cut(fb['floor'], [-10, -0.5, 0.5, 1.5, 3.5, 7.5, 200],
                          labels=['Basement', 'Ground', '1st', '2–3', '4–7', '8+'])
ct = pd.crosstab(fb['landuse'], fb['floor_band'], normalize='index').reindex(LANDUSE_ORDER).dropna(how='all')
n  = fb['landuse'].value_counts().reindex(ct.index)

fig, ax = plt.subplots(figsize=(9, 0.45 * len(ct) + 1.6))
cmap = mpl.colors.LinearSegmentedColormap.from_list('seq', SEQ)
im = ax.imshow(ct.values, cmap=cmap, vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(ct.shape[1]), ct.columns); ax.set_yticks(range(len(ct)), [f'{i}  (n={n[i]:,})' for i in ct.index])
ax.tick_params(colors=INK2, length=0); ax.grid(False)
for s in ax.spines.values(): s.set_visible(False)
for i in range(ct.shape[0]):
    for j in range(ct.shape[1]):
        v = ct.values[i, j]
        if v >= 0.05:
            ax.text(j, i, f'{v:.0%}', ha='center', va='center', fontsize=8.5, color='white' if v > 0.45 else INK)
ax.set_title('Floor of POIs by land use (row = 100%)')
cb = fig.colorbar(im, ax=ax, pad=0.01, format=mpl.ticker.PercentFormatter(1)); cb.outline.set_visible(False)
fig.tight_layout(); save(fig, 'p3_landuse_by_floor.png'); plt.show()

### P4. Where each land use is (small multiples)

In [ ]:
P = poi.to_crs(UTM)
uses = [u for u in LANDUSE_ORDER if u in P['landuse'].values][:9]
ncol = 3; nrow = int(np.ceil(len(uses) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(12, 4 * nrow))
boundary = aoi.to_crs(UTM).boundary
for ax, u in zip(axes.flat, uses):
    boundary.plot(ax=ax, color=AXIS, linewidth=0.6)
    ax.scatter(P.geometry.x, P.geometry.y, s=0.4, color=GRID, linewidths=0, rasterized=True)          # all POIs, faint
    sub = P[P['landuse'] == u]
    ax.scatter(sub.geometry.x, sub.geometry.y, s=1.2, color=BLUE, linewidths=0, rasterized=True)
    ax.set_title(f'{u}  ({len(sub):,})', fontsize=10); ax.set_axis_off()
for ax in axes.flat[len(uses):]:
    ax.set_axis_off()
fig.suptitle(f'{AREA_NAME.title()}: POIs by land use', x=0.01, ha='left', fontsize=13, fontweight='bold')
fig.tight_layout(rect=[0, 0, 1, 0.97]); save(fig, 'p4_landuse_maps.png'); plt.show()

### P5. Ward summary
POI density and how many POIs have a known floor, per ward. The table is saved as CSV for the socio-economic analysis.

In [ ]:
if 'ward' in poi.columns:
    wards = aoi[[WARD_COL, 'geometry']].copy()
    wards['area_km2'] = wards.to_crs(UTM).area / 1e6
    s = poi.groupby('ward').agg(pois=('landuse', 'size'), floor_known=('floor', lambda x: x.notna().mean()),
                                offices=('landuse', lambda x: (x == 'Office').sum()))
    wards = wards.merge(s, left_on=WARD_COL, right_index=True, how='left')
    wards['pois_per_km2'] = wards['pois'] / wards['area_km2']
    wards.drop(columns='geometry').to_csv(os.path.join(OUT_DIR, f'ward_poi_summary_{AREA_NAME}.csv'), index=False)

    cmap = mpl.colors.LinearSegmentedColormap.from_list('seq', SEQ)
    fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))
    for ax, col, title, fmt in [(axes[0], 'pois_per_km2', 'POIs per km²', '{:,.0f}'),
                                (axes[1], 'floor_known', 'Share of POIs with a known floor', '{:.0%}')]:
        vmax = wards[col].quantile(0.97)
        wards.plot(column=col, ax=ax, cmap=cmap, vmin=0, vmax=vmax, edgecolor=SURFACE, linewidth=0.4,
                   missing_kwds={'color': NODATA, 'edgecolor': SURFACE, 'linewidth': 0.4})
        ax.set_axis_off(); ax.set_title(title)
        sm = mpl.cm.ScalarMappable(cmap=cmap, norm=mpl.colors.Normalize(0, vmax))
        cb = fig.colorbar(sm, ax=ax, shrink=0.6, pad=0.01); cb.outline.set_visible(False)
        cb.ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _, f=fmt: f.format(v)))
    fig.suptitle(f'{AREA_NAME.title()}: {len(wards)} wards', x=0.01, ha='left', fontsize=13, fontweight='bold')
    fig.tight_layout(rect=[0, 0, 1, 0.96]); save(fig, 'p5_ward_summary.png'); plt.show()
    display(wards.drop(columns='geometry').sort_values('pois', ascending=False).head(10).round(2))